# Explore YOLO11 for Face Detection

In this notebook we use a **pre-trained YOLO11n model** (trained on the
[WIDERFACE](http://shuoyang1213.me/WIDERFACE/) dataset) to detect faces
in images, draw bounding boxes, and inspect the raw detection output.

The model is hosted on HuggingFace at
[AdamCodd/YOLOv11n-face-detection](https://huggingface.co/AdamCodd/YOLOv11n-face-detection).

**Goal:** confirm that YOLO11 can locate faces — but note it only detects
the generic class `face`. It cannot identify *whose* face it is.
We will train it to distinguish specific individuals in notebook 02.

In [ ]:
!pip install --no-cache-dir -q -r requirements.txt

import glob
from ultralytics import YOLO
from huggingface_hub import hf_hub_download
from PIL import Image
from IPython.display import display

In [ ]:
model_path = hf_hub_download(
    repo_id="AdamCodd/YOLOv11n-face-detection",
    filename="model.pt"
)
model = YOLO(model_path)

## Detect a single face

Run YOLO11 on a portrait photo and inspect what it found.

In [ ]:
results = model.predict("images/test_face_01.jpg", conf=0.6)
result = results[0]

print(f"Faces detected: {len(result.boxes)}")
print(f"Class names: {result.names}")

box = result.boxes[0]
cords = [round(x) for x in box.xyxy[0].tolist()]
conf = round(box.conf[0].item(), 3)
print(f"\nDetection: class={result.names[int(box.cls[0].item())]}, confidence={conf}, box={cords}")

The model found 1 face with class `face` (ID 0). It knows *something is a face*
but has no idea *whose* face it is. After retraining in notebook 02, the class
map will become `{0: 'adnan', 1: 'unknown_face'}`.

## Visualize with bounding boxes

In [ ]:
Image.fromarray(result.plot()[:, :, ::-1])

## Detect faces in group photos

Now test on images with multiple people. The model should detect every visible face.

In [ ]:
group_images = sorted(glob.glob("images/test_group_*.jpg"))
print(f"Testing {len(group_images)} group images\n")

for img_path in group_images:
    results = model.predict(img_path)
    result = results[0]

    print(f"=== {img_path} ===")
    for box in result.boxes:
        class_id = result.names[box.cls[0].item()]
        cords = [round(x) for x in box.xyxy[0].tolist()]
        conf = round(box.conf[0].item(), 2)
        print(f"  {class_id}: conf={conf}  box={cords}")

    display(Image.fromarray(result.plot()[:, :, ::-1]))
    print()

## Summary

YOLO11 reliably detects faces in both portraits and group photos. Every
detection is labelled as the generic class `face` — the model cannot tell
people apart.

**Next:** Open **`02-retrain-face-model.ipynb`** to train the model to
recognize your face specifically.